In [ ]:
import math
import warnings

import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq

warnings.filterwarnings('ignore')

# GARCH Tail Index
- Define Tail Index Function
- Minimise to Find Tail Index for Fixed Alpha and Beta


## Tail Index Function

$$
f(x) = \mathbb{E}((\alpha_1Z^2 + \beta_1)^\frac{x}{2}) - 1 = \beta_1^\frac{x}{2} + 2\int^\infty_{\beta_1^\frac{x}{2}} F_Z\left( -\sqrt{\frac{t^\frac{2}{x} - \beta_1}{\alpha_1}} \right) \, dt - 1
$$

In [16]:
# Find Expectation
def integrand(x, k, dist, alpha, beta):
        return 2 * dist.cdf(-np.sqrt((x**(2/k) - beta)/alpha))

def expectation(k, dist, alpha, beta):
    value, error = quad(integrand, beta**(k/2), math.inf, args=(k,dist,alpha,beta), epsabs=1e-8)
    return beta**(k/2) + value

def fn(k, dist, alpha, beta):
        return expectation(k, dist, alpha, beta) - 1

## Find Tail Index

In [ ]:
def tail_index(dist, alpha, beta):
    '''Compute Tail Index for Given Alpha and Beta'''
    low = 0.1
    high =  5.1
    while True:
        try:
            ti = brentq(fn, low, high, xtol=1e-8, args=(dist, alpha, beta))
            if alpha+beta > 1 and ti > 10:
                return np.nan
            return ti
        except:
            low += 5
            high += 5
            if high > 100:
                if alpha + beta > 1:
                    return np.nan
                return 100

## Find Alpha from TI

In [5]:
def alpha_fn(alpha, dist, ti, beta):
    try:
        return tail_index(dist, alpha, beta) - ti
    except:
        return 10**10

In [6]:
def find_alpha(ti, dist, beta):
    alpha = np.nan
    alpha = brentq(alpha_fn, 0, 1-beta, xtol=1e-8, args=(dist, ti, beta))
    return alpha